# Chấm sentiment: VADER và ghép điểm transformer

So với bản cũ:

- Văn bản chấm điểm là "tiêu đề tự viết. nội dung", không còn chứa tiêu đề tự sinh kiểu "Five Stars" (đã tách ở `prepare_and_eda`).
- Bỏ phần TF-IDF + hồi quy logistic: mô hình đó được huấn luyện để đoán `rating == 5`, nên điểm của nó là "học sao rồi dự đoán sao", không phải sentiment độc lập với số sao.
- `has_text` đổi thành `has_signal`: review có tiêu đề tự viết hoặc nội dung. Review không có chữ nào sẽ nhận sentiment trung tính ở bước hybrid.
- VADER chỉ giữ làm đường cơ sở để so với transformer. Phân tích trước đó cho thấy VADER yếu trên review âm nhạc, nhiều khả năng cũng yếu trên review game.

Điểm transformer (`tf01`) được tạo bằng `python -m src.sentiment_transformer score` (xem notebook `sentiment_transformer`), rồi ghép vào `sentiment_all.csv.gz` ở cuối notebook này.

In [7]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.config import CATEGORY, CORE_FILE, DATA, SENT_FILE
from src.sentiment_transformer import load_docs, make_docs, diagnose, TF_FILE

df = load_docs()
df["n_words"] = df.text.str.split().str.len().fillna(0).astype(int)
df["doc"] = make_docs(df)
df["has_signal"] = df.doc.str.strip() != ""
print(CATEGORY, "| số review:", f"{len(df):,}", "| có chữ:", round(df.has_signal.mean(), 3))

Video_Games | số review: 257,475 | có chữ: 1.0


## 1. VADER (đường cơ sở)

In [8]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError as e:
    raise SystemExit("Cài thư viện: %pip install vaderSentiment") from e

an = SentimentIntensityAnalyzer()
df["vader"] = [an.polarity_scores(t)["compound"] if t.strip() else np.nan for t in df.doc]   # trong [-1, 1]
df["vader01"] = (df.vader + 1) / 2

print(df.groupby("rating").vader.agg(["mean", "std", "size"]).round(3))
print("4-5 sao nhưng vader < -0.05:", round(((df.rating >= 4) & (df.vader < -0.05)).mean(), 3))
print("1-2 sao nhưng vader > 0.05:", round(((df.rating <= 2) & (df.vader > 0.05)).mean(), 3))
df[["user_id", "parent_asin", "vader", "has_signal"]].to_csv(DATA / "sentiment_vader.csv.gz", index=False)

         mean    std    size
rating                      
1.0    -0.150  0.676   14005
2.0     0.087  0.739   11080
3.0     0.400  0.654   23652
4.0     0.670  0.484   45346
5.0     0.708  0.362  163392
4-5 sao nhưng vader < -0.05: 0.038
1-2 sao nhưng vader > 0.05: 0.042


Xem bằng mắt: trong nhóm 5 sao, VADER đang phân biệt điều gì? Chú ý các câu bị chấm sai do luật phủ định, câu châm biếm, hoặc từ vựng game ("kill", "dark", "hard", "grind"...).

In [9]:
five = df[(df.rating == 5) & df.has_signal].sort_values("vader")
for name, part in [("5 SAO, VADER THẤP NHẤT", five.head(10)), ("5 SAO, VADER CAO NHẤT", five.tail(10))]:
    print("\n=====", name)
    for _, r in part.iterrows():
        print(f"[{r.vader:+.2f}]", r.doc[:200].replace("\n", " "))


===== 5 SAO, VADER THẤP NHẤT
[-1.00] MY FAVORITE TELLTALE GAMES SERIES Minecraft: Story Mode. Minecraft: Story Mode - Season 2 - PlayStation 4 Standard Edition IS THE BEST BUT I'M 2 SAD ABOUT  Minecraft: Story Mode BE PULLED FROM STORES 
[-1.00] MY FAVORITE TELLTALE GAMES SERIES Minecraft: Story Mode. I SO SAD 4 MY FAVORITE TELLTALE GAMES SERIES Minecraft: Story Mode PULLED OF STORES LIKE BOO WHO WHO 😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢😢
[-1.00] A fun game. Here are the statsic for the first two boss if you due gutsman  and cut man. It will take about 14 hits to take out cut man if you due not use his weakness bomb man ability and for cut man
[-1.00] Castlevania Lords of Shadows Pt 3 Dracula/Gabriel's Redemption. Castlevania Lords Of Shadows 2<br />Now we have come to the last game of the supposed Lords Of Shadows Trilogy and it's good but maybe s
[-1.00] Reviving Evil. In 2005, Resident Evil 4 came out and blew the entire gaming world away. At the time of its release there was n

## 2. Tạo `sentiment_all.csv.gz`

Cột `vader01` luôn có. Cột `tf01` được ghép nếu đã chạy `python -m src.sentiment_transformer score`.

In [10]:
sa = df[["user_id", "parent_asin", "vader01", "has_signal"]].copy()
if TF_FILE.exists():
    tf = pd.read_csv(TF_FILE)
    sa = sa.merge(tf[["user_id", "parent_asin", "tf01"]], on=["user_id", "parent_asin"], validate="1:1")
    print("đã ghép tf01")
else:
    print("Chưa có", TF_FILE.name, "-> chạy: python -m src.sentiment_transformer score")
sa.to_csv(SENT_FILE, index=False)
print("đã lưu", SENT_FILE)

đã ghép tf01
đã lưu D:\PycharmProjects\Sentiment_fix\data\processed\Video_Games\sentiment_all.csv.gz


## 3. So sánh hai điểm sentiment

Hai cách đọc cần đặt cạnh nhau: toàn bộ review, và chỉ các review đủ dài (15 từ trở lên). Nhóm sau loại bỏ những review quá ngắn mà mô hình dễ chấm sai, nên gần với tín hiệu thật của bình luận hơn.

In [11]:
d = df.merge(sa[["user_id", "parent_asin"] + [c for c in ("tf01",) if c in sa.columns]],
             on=["user_id", "parent_asin"], validate="1:1")
for col in [c for c in ("vader01", "tf01") if c in d.columns]:
    diagnose(d, col)
    diagnose(d, col, min_words=15)


== vader01 (review xét: 257,447)
         mean    std    size
rating                      
1.0     0.425  0.338   14005
2.0     0.544  0.369   11080
3.0     0.700  0.327   23652
4.0     0.835  0.242   45346
5.0     0.854  0.181  163364
đơn điệu tăng theo sao: True
AUC (1-2 sao vs 5 sao): 0.792
AUC (<=3 sao vs >=4 sao): 0.702
Spearman với sao: 0.204
Spearman chỉ trong nhóm 1-4 sao: 0.42
trong nhóm 5 sao: trung bình 0.854, độ lệch chuẩn 0.181
tỉ lệ bão hòa (>0.9): 0.529

== vader01 (review xét: 176,534, >= 15 từ)
         mean    std   size
rating                     
1.0     0.434  0.356  11910
2.0     0.555  0.381   9985
3.0     0.717  0.342  20147
4.0     0.858  0.250  37026
5.0     0.901  0.185  97466
đơn điệu tăng theo sao: True
AUC (1-2 sao vs 5 sao): 0.829
AUC (<=3 sao vs >=4 sao): 0.737
Spearman với sao: 0.314
Spearman chỉ trong nhóm 1-4 sao: 0.44
trong nhóm 5 sao: trung bình 0.901, độ lệch chuẩn 0.185
tỉ lệ bão hòa (>0.9): 0.668

== tf01 (review xét: 257,447)
         mean    s